# Train SwinUNETR iteratively on blob pseudo-labels (DDeep3M+ template)

Iterative weakly-supervised fine-tuning of **SwinUNETR** for binary synapse
segmentation on 2D 128x128 fluorescence patches. Each outer round of the
loop alternates four steps adapted from Xiao et al., *DDeep3M+: adaptive
enhancement powered weakly supervised learning for neuron segmentation*,
Neurophotonics 10(3), 035003, 2023 (PMC10289179):

1. **Pseudo-label generation** (paper step 1, ours: `pseudolabels.blobs`)
   - LoG blobs on pre/post synaptic channels, co-localised and gated by a
     Meijering ridge + soma structural mask.
2. **Train SwinUNETR** for `epochs_per_iter` epochs on `(image, pseudo)`
   pairs with Dice+BCE loss (paper step 2).
3. **Region-grow** the previous pseudo-label using the network's
   probability map and an adaptive ring-mean threshold
   (`pseudolabels.refine.region_grow_from_prob`; paper Eq. 5-6, 2D).
4. **Fuse** the raw image with the probability map and re-run the blob
   pipeline on the fused image
   (`pseudolabels.refine.fuse_image_with_prob`; paper Eq. 7, our reading).

The new pseudo-label is `grown | blob_new`, structurally gated.

**Documented deviations from the paper** (see `pseudolabels/refine.py`
module docstring for full reasoning):
- 2D, not 3D. Patch size 128x128.
- Only synaptic channels (pre+post) are fused with the probability map;
  the structural/neurite channel is left raw to keep the anatomical
  prior stable.
- The structural mask is computed per source image once, then sliced
  per patch and reused across iterations (the structural channel is
  not modified by fusion).
- Stopping criterion: self-consistency of pseudo-labels between rounds
  (mean per-patch IoU + positive-pixel fraction guards), since no
  manual ground truth is available. The paper stops on held-out F1
  plateau.

## Imports

In [ ]:
import os, sys, json, time, copy
from collections import defaultdict
from datetime import datetime
from pathlib import Path

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

## Path setup

In [ ]:
NB_DIR = Path.cwd().resolve()
# Walk up to repo root (folder containing .git/)
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / '.git').is_dir():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / 'root'   # source dir (Python packages)
for p in (ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('REPO_ROOT:', REPO_ROOT)
print('ROOT     :', ROOT)

In [ ]:
# Training infrastructure
from training.config import BaseCfg, DataCfg, ModelCfg, dump_config
from training.seeding import seed_everything
from training.logging import setup_logger, CSVMetricLogger
from training.data import compute_channel_stats
from training.lr_schedule import param_groups_layer_decay, make_warmup_cosine
from training.checkpoints import save_checkpoint, load_checkpoint, find_latest_checkpoint

# Pseudo-labels: classical (iter 0) + refinement (iter >= 1)
from pseudolabels import (
    BlobPseudoCfg, generate_blob_pseudolabel,
    compute_global_meijering_threshold,
    compute_fullimage_structural_mask,
    RefineCfg, region_grow_from_prob, fuse_image_with_prob,
    compute_mask_iou, compute_positive_fraction,
)

# Segmentation
from segmentation import (
    SegTrainCfg,
    PseudoLabelSegDataset,
    DiceBCELoss, compute_dice_metric,
    build_swinunetr, load_pretrained_encoder_into_swinunetr, count_params,
    SegTrainTransform, SegValTransform,
    sliding_window_predict,
    plot_seg_overlay, plot_seg_comparison, plot_seg_curves,
    plot_full_image_result,
)

from utils_data.patch_dataset import PatchDataset
from utils_data.reassemble import reassemble_image, list_image_indices

## Configuration

In [ ]:
base_cfg = BaseCfg(
    seed                 = 42,
    output_root          = '../outputs',
    experiment_name      = 'swinunetr_seg_pseudolabels_iter',
    tag                  = 'ddeep3m_plus',
    method_name          = 'swinunetr_seg_iterative',
    init_source          = 'local_ckpt',
    pretrained_ckpt_path = None,  # SET THIS to your pretrained encoder .pt
    resume_path          = None,
    dry_run              = False,
)

In [ ]:
data_cfg = DataCfg(
    data_root        = '../../data/patches_128',
    exclude_patterns = ['KONTROLA'],
    val_split        = 0.15,
    batch_size       = 16,
    num_workers      = 2,
    pin_memory       = True,
    channel_names    = ['pre_synaptic', 'post_synaptic', 'structural'],
)

In [ ]:
model_cfg = ModelCfg(
    in_channels       = 3,
    img_size          = 128,
    feature_size      = 96,
    patch_size        = 2,
    window_size       = 7,
    depths            = (2, 2, 6, 2),
    num_heads         = (3, 6, 12, 24),
    dropout_path_rate = 0.1,
)

In [ ]:
# `epochs` here means epochs per OUTER iteration.
# Total compute budget = epochs_per_iter * max_iters (set below).
seg_cfg = SegTrainCfg(
    epochs                = 25,
    warmup_epochs         = 2,
    base_lr               = 1e-4,
    decoder_lr            = 5e-4,
    weight_decay          = 0.05,
    layer_decay           = 0.75,
    grad_clip_norm        = 5.0,
    freeze_encoder_epochs = 3,
    save_every_n_epochs   = 25,
    val_metric_key        = 'dice',
    val_metric_direction  = 'max',
    model_save_name       = 'swinunetr_seg_iter_best.pt',
    dice_weight           = 1.0,
    bce_weight            = 1.0,
    dice_smooth           = 1.0,
    pseudolabel_cache_dir = None,  # IMPORTANT: disabled to avoid stale-mask reuse
)

In [ ]:
pseudo_cfg = BlobPseudoCfg(
    log_min_sigma     = 0.7,
    log_max_sigma     = 1.8,
    log_num_sigma     = 5,
    log_threshold     = 0.005,
    coloc_dilation    = 2,
    dendrite_threshold = None,  # computed below
    use_zscore        = True,
    zscore_threshold  = 5.0,
)

In [ ]:
refine_cfg = RefineCfg(
    ring_radius              = 2,
    growth_connectivity      = 8,
    max_grow_iters           = 10,
    rho_min                  = 0.30,
    rho_max                  = 0.95,
    max_growth_ratio         = 5.0,
    gate_growth_by_structural = True,
    prob_threshold           = 0.30,
    weight_raw               = 0.80,
    weight_prob              = 0.20,
    fuse_channels            = (0, 1),  # pre + post synaptic only
    intensity_max            = 1.0,
)

In [ ]:
# Outer-loop controls
from dataclasses import dataclass

@dataclass
class IterCfg:
    max_iters: int = 5
    # Stop when (1 - mean IoU) < eps AND positive-fraction change small.
    stop_iou_eps: float = 0.005
    stop_pos_frac_eps: float = 0.005
    # Sanity bounds on positive-pixel fraction; if labels collapse to
    # empty or exceed this, do NOT stop (force continued training).
    pos_frac_min: float = 1e-4
    pos_frac_max: float = 0.20
    # Re-instantiate the model each iteration (cleaner but ~max_iters times
    # more compute). Default: warm-start across iterations.
    reinit_model_per_iter: bool = False
    # Only freeze the encoder during iteration 1 (paper-style transfer warm-up).
    freeze_encoder_iter1_only: bool = True

iter_cfg = IterCfg()

## Output directory + logger

In [ ]:
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = Path(base_cfg.output_root) / f'{base_cfg.experiment_name}_{base_cfg.tag}_{RUN_TS}'
save_dir.mkdir(parents=True, exist_ok=True)
print('save_dir =', save_dir)

logger = setup_logger(f'seg_iter_{RUN_TS}', save_dir / 'run.log')
logger.info(f'save_dir = {save_dir}')

dump_config(
    save_dir / 'config.json',
    base=base_cfg, data=data_cfg, model=model_cfg, seg=seg_cfg,
    pseudo=pseudo_cfg, refine=refine_cfg, iter_=iter_cfg,
)
seed_everything(base_cfg.seed)

## Data + channel statistics

In [ ]:
# Single canonical PatchDataset shared by every iteration.
patch_ds = PatchDataset(
    data_cfg.data_root,
    exclude_patterns=data_cfg.exclude_patterns,
)
N = len(patch_ds)
logger.info(f'patches: N={N}  channels={patch_ds.num_channels}  size={patch_ds.patch_size}')

# Deterministic train/val split by indices.
rng = np.random.default_rng(base_cfg.seed)
perm = rng.permutation(N)
n_val = max(1, int(round(N * data_cfg.val_split)))
val_indices = sorted(perm[:n_val].tolist())
train_indices = sorted(perm[n_val:].tolist())
logger.info(f'split: {len(train_indices)} train / {len(val_indices)} val')

In [ ]:
ch_mean, ch_std = compute_channel_stats(
    patch_ds,
    in_channels=model_cfg.in_channels,
    max_samples=data_cfg.channel_stats_max_samples,
)
ch_mean_t = ch_mean.float() if isinstance(ch_mean, torch.Tensor) else torch.tensor(ch_mean, dtype=torch.float32)
ch_std_t  = ch_std.float()  if isinstance(ch_std, torch.Tensor)  else torch.tensor(ch_std,  dtype=torch.float32)
logger.info(f'channel mean={ch_mean_t.tolist()}, std={ch_std_t.tolist()}')
(save_dir / 'channel_stats.json').write_text(json.dumps({
    'channel_names': list(data_cfg.channel_names),
    'mean': ch_mean_t.tolist(),
    'std':  ch_std_t.tolist(),
}, indent=2))

## Per-image structural masks

Group patches by `image_index`, reassemble each full image, and run
Meijering + soma detection on the full image. Slice the resulting
`near_structural` per patch. The structural channel (index 2) is never
fused, so these slices stay valid across iterations and we only pay
the cost once.

In [ ]:
# Collect a global Meijering threshold from a sample of structural
# channels (avoids per-patch Otsu instability on neurite-free patches).
sample_idx = rng.choice(N, size=min(256, N), replace=False)
struct_channel = pseudo_cfg.structural_channel
sample_patches = []
for idx in sample_idx:
    rec = patch_ds.records[int(idx)]
    arr = np.load(patch_ds.root / rec['filename']).astype(np.float32)
    sample_patches.append(arr[struct_channel])
pseudo_cfg.dendrite_threshold = compute_global_meijering_threshold(
    sample_patches,
    sigmas=pseudo_cfg.dendrite_sigmas,
    method='otsu',
)
logger.info(f'global Meijering threshold = {pseudo_cfg.dendrite_threshold:.6f}')

In [ ]:
# Group patches by image_index; only consider images that contribute >=1 patch.
by_image_idx = defaultdict(list)
for rec in patch_ds.records:
    by_image_idx[int(rec['image_index'])].append(rec)
image_indices = sorted(by_image_idx.keys())
logger.info(f'{len(image_indices)} source images contribute {N} patches total')

struct_slices: dict[str, dict] = {}  # filename -> {'structural_mask': ..., 'near_structural': ...}
patch_size = int(patch_ds.patch_size)

for img_idx in tqdm(image_indices, desc='per-image structural masks'):
    try:
        full_image, recs = reassemble_image(
            data_cfg.data_root, img_idx,
            exclude_patterns=data_cfg.exclude_patterns,
        )
    except ValueError as e:
        logger.warning(f'skipping image {img_idx}: {e}')
        continue
    struct = compute_fullimage_structural_mask(full_image, pseudo_cfg)
    for rec in recs:
        r = int(rec['grid_row']); c = int(rec['grid_col'])
        y0, x0 = r * patch_size, c * patch_size
        struct_slices[rec['filename']] = {
            'structural_mask': struct['structural_mask'][y0:y0+patch_size, x0:x0+patch_size].copy(),
            'near_structural': struct['near_structural'][y0:y0+patch_size, x0:x0+patch_size].copy(),
        }

# All records the dataset will iterate over must have a structural slice.
missing = [r['filename'] for r in patch_ds.records if r['filename'] not in struct_slices]
assert not missing, f'missing structural slices for {len(missing)} patches; first: {missing[:3]}'
logger.info(f'structural slices: {len(struct_slices)}')

## Iteration 0 pseudo-labels (classical blob pipeline)

In [ ]:
def _load_patch(idx: int) -> np.ndarray:
    rec = patch_ds.records[idx]
    arr = np.load(patch_ds.root / rec['filename']).astype(np.float32)
    if patch_ds.channels is not None:
        arr = arr[patch_ds.channels]
    return arr

current_pseudo: dict[str, np.ndarray] = {}
for idx in tqdm(range(N), desc='iter-0 pseudo-labels'):
    rec = patch_ds.records[idx]
    fname = rec['filename']
    patch = _load_patch(idx)
    mask, _interm, _stats = generate_blob_pseudolabel(
        patch, pseudo_cfg, precomputed_struct=struct_slices[fname],
    )
    current_pseudo[fname] = mask.astype(np.uint8)

pos_fracs = [compute_positive_fraction(m) for m in current_pseudo.values()]
logger.info(
    f'iter-0 pseudo-labels: mean positive fraction = {float(np.mean(pos_fracs)):.5f} '
    f'(min={float(np.min(pos_fracs)):.5f}, max={float(np.max(pos_fracs)):.5f})'
)

## Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def build_model():
    m = build_swinunetr(model_cfg, out_channels=1).to(device)
    if base_cfg.pretrained_ckpt_path is not None:
        load_pretrained_encoder_into_swinunetr(
            m, base_cfg.pretrained_ckpt_path, logger=logger,
        )
    else:
        logger.warning('No pretrained_ckpt_path set; encoder starts random.')
    return m

model = build_model()
logger.info(f'model params: {count_params(model)/1e6:.2f}M')

## Iteration helpers

In [ ]:
def build_loaders(current_pseudo_dict):
    # Wrap PatchDataset with current pseudo-labels and split into loaders.
    # cache_dir=None is critical: it prevents the dataset from silently
    # loading stale .npy masks from any previous run.
    fnames = {r['filename'] for r in patch_ds.records}
    missing = fnames - set(current_pseudo_dict)
    assert not missing, f'pseudo-labels missing for {len(missing)} patches'

    ds_train = PseudoLabelSegDataset(
        patch_ds, pseudo_cfg,
        cache_dir=None,
        transform=SegTrainTransform(ch_mean_t, ch_std_t),
        precomputed_masks=current_pseudo_dict,
    )
    ds_val = PseudoLabelSegDataset(
        patch_ds, pseudo_cfg,
        cache_dir=None,
        transform=SegValTransform(ch_mean_t, ch_std_t),
        precomputed_masks=current_pseudo_dict,
    )
    train_loader = DataLoader(
        Subset(ds_train, train_indices),
        batch_size=data_cfg.batch_size,
        shuffle=True,
        num_workers=data_cfg.num_workers,
        pin_memory=data_cfg.pin_memory,
        drop_last=True,
    )
    val_loader = DataLoader(
        Subset(ds_val, val_indices),
        batch_size=data_cfg.batch_size,
        shuffle=False,
        num_workers=data_cfg.num_workers,
        pin_memory=data_cfg.pin_memory,
    )
    return train_loader, val_loader

In [ ]:
def train_iteration(model, train_loader, val_loader, *, iter_idx, freeze_encoder, csv_logger):
    # Run seg_cfg.epochs epochs on the current pseudo-labels.
    # Cosine LR schedule is reset per outer iteration so that warm-up
    # happens once at the start of each block; document this in the
    # thesis (one global cosine across iterations is the alternative).
    loss_fn = DiceBCELoss(
        dice_weight=seg_cfg.dice_weight,
        bce_weight=seg_cfg.bce_weight,
        dice_smooth=seg_cfg.dice_smooth,
    )
    # Encoder: layer-decay LR groups. Decoder: flat LR.
    encoder_groups = param_groups_layer_decay(
        model.swinViT,
        base_lr=seg_cfg.base_lr,
        weight_decay=seg_cfg.weight_decay,
        layer_decay=seg_cfg.layer_decay,
    )
    encoder_param_ids = {id(p) for p in model.swinViT.parameters()}
    decoder_params = [p for p in model.parameters() if id(p) not in encoder_param_ids]
    decoder_group = {
        'params': decoder_params,
        'lr': seg_cfg.decoder_lr,
        'weight_decay': seg_cfg.weight_decay,
    }
    optimizer = torch.optim.AdamW(
        encoder_groups + [decoder_group], betas=(0.9, 0.999),
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, make_warmup_cosine(seg_cfg.warmup_epochs, seg_cfg.epochs),
    )
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == 'cuda')

    def set_encoder_grad(flag: bool):
        for p in model.swinViT.parameters():
            p.requires_grad = flag

    best_val_dice = -1.0
    best_state = None

    for epoch in range(1, seg_cfg.epochs + 1):
        in_freeze = bool(freeze_encoder and epoch <= seg_cfg.freeze_encoder_epochs)
        set_encoder_grad(not in_freeze)
        phase = 'frozen' if in_freeze else 'full'

        # ----- train -----
        model.train()
        t_train = time.time()
        run_loss, run_dl, run_bl = 0.0, 0.0, 0.0
        grads = []
        pbar = tqdm(
            train_loader,
            desc=f'iter {iter_idx} ep {epoch}/{seg_cfg.epochs} [{phase}]',
            leave=False,
        )
        for img_b, mask_b in pbar:
            img_b = img_b.to(device, non_blocking=True)
            mask_b = mask_b.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device.type, enabled=device.type == 'cuda'):
                logits = model(img_b)
                losses = loss_fn(logits, mask_b)
            scaler.scale(losses['loss']).backward()
            scaler.unscale_(optimizer)
            gn = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=seg_cfg.grad_clip_norm,
            )
            scaler.step(optimizer)
            scaler.update()
            run_loss += float(losses['loss'].item())
            run_dl   += float(losses['dice_loss'].item())
            run_bl   += float(losses['bce_loss'].item())
            grads.append(float(gn))
        scheduler.step()
        n_batches = max(1, len(train_loader))
        train_loss = run_loss / n_batches
        train_dl   = run_dl / n_batches
        train_bl   = run_bl / n_batches
        train_time = time.time() - t_train

        # ----- val -----
        model.eval()
        t_val = time.time()
        val_loss_sum, val_dice_sum, n_val = 0.0, 0.0, 0
        with torch.no_grad():
            for img_b, mask_b in val_loader:
                img_b = img_b.to(device, non_blocking=True)
                mask_b = mask_b.to(device, non_blocking=True)
                with torch.amp.autocast(device.type, enabled=device.type == 'cuda'):
                    logits = model(img_b)
                    vl = loss_fn(logits, mask_b)
                val_loss_sum += float(vl['loss'].item()) * img_b.size(0)
                # compute_dice_metric takes LOGITS (applies sigmoid internally).
                val_dice_sum += float(
                    compute_dice_metric(logits, mask_b, threshold=0.5).item()
                ) * img_b.size(0)
                n_val += img_b.size(0)
        val_loss = val_loss_sum / max(1, n_val)
        val_dice = val_dice_sum / max(1, n_val)
        val_time = time.time() - t_val

        enc_lr = [g['lr'] for g in optimizer.param_groups if g.get('stage') is not None]
        dec_lr = [g['lr'] for g in optimizer.param_groups if g.get('stage') is None]
        csv_logger.log({
            'iter_idx': iter_idx,
            'epoch': epoch,
            'phase': phase,
            'train_loss': train_loss,
            'train_dice_loss': train_dl,
            'train_bce_loss': train_bl,
            'val_loss': val_loss,
            'val_dice': val_dice,
            'lr_encoder': max(enc_lr) if enc_lr else 0.0,
            'lr_decoder': dec_lr[0] if dec_lr else 0.0,
            'epoch_time_s': train_time + val_time,
            'train_time_s': train_time,
            'val_time_s': val_time,
            'grad_norm_mean': float(np.mean(grads)) if grads else 0.0,
            'grad_norm_max':  float(np.max(grads))  if grads else 0.0,
        })

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val_dice

In [ ]:
@torch.no_grad()
def predict_per_patch_probs(model) -> dict[str, np.ndarray]:
    # Return {filename -> (H, W) prob_map} via batched eval forward.
    model.eval()
    out: dict[str, np.ndarray] = {}
    bs = data_cfg.batch_size
    ch_mean_np = ch_mean_t.view(-1, 1, 1).numpy()
    ch_std_np  = ch_std_t.view(-1, 1, 1).numpy()
    batch_pos: list[tuple[int, str]] = []
    batch_arr: list[np.ndarray] = []

    def flush():
        if not batch_arr:
            return
        x = torch.from_numpy(np.stack(batch_arr)).float().to(device)
        with torch.amp.autocast(device.type, enabled=device.type == 'cuda'):
            logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()[:, 0]
        for (_idx, fname), p in zip(batch_pos, probs):
            out[fname] = p.astype(np.float32)
        batch_pos.clear()
        batch_arr.clear()

    for idx in tqdm(range(N), desc='per-patch prob_map'):
        rec = patch_ds.records[idx]
        arr = _load_patch(idx)
        normed = (arr - ch_mean_np) / ch_std_np
        batch_pos.append((idx, rec['filename']))
        batch_arr.append(normed)
        if len(batch_arr) >= bs:
            flush()
    flush()
    return out

In [ ]:
def refine_pseudo_labels(current_pseudo_dict, prob_maps):
    # DDeep3M+ steps 3 + 4: region-grow + image-prob fusion -> new blobs.
    # Returns (new_pseudo_dict, diagnostics_dict).
    new_pseudo: dict[str, np.ndarray] = {}
    ious, pos_fracs, grow_added = [], [], []
    for idx in tqdm(range(N), desc='refine pseudo-labels'):
        rec = patch_ds.records[idx]
        fname = rec['filename']
        prev_mask = current_pseudo_dict[fname]
        prob = prob_maps[fname]
        slc = struct_slices[fname]
        patch = _load_patch(idx)

        # Step 3: region growing on prev mask, gated by structural prior.
        grown = region_grow_from_prob(
            prev_mask, prob, refine_cfg,
            structural_mask=slc['near_structural'],
        )

        # Step 4: image-probability fusion on pre+post channels only.
        fused = fuse_image_with_prob(patch, prob, refine_cfg)

        # Re-run the blob pipeline on the fused image (structural mask
        # is unchanged because we did not fuse the structural channel,
        # so we pass the cached structural intermediates).
        blob_new, _interm, _stats = generate_blob_pseudolabel(
            fused, pseudo_cfg, precomputed_struct=slc,
        )

        # New pseudo-label = grown U blob_new. Both are structurally
        # gated already (grown via cfg, blob_new via generate_*).
        new_mask = np.logical_or(grown.astype(bool), blob_new.astype(bool)).astype(np.uint8)

        new_pseudo[fname] = new_mask
        ious.append(compute_mask_iou(prev_mask, new_mask))
        pos_fracs.append(compute_positive_fraction(new_mask))
        grow_added.append(int(grown.sum()) - int(prev_mask.astype(bool).sum()))

    diag = {
        'mean_iou_vs_prev': float(np.mean(ious)),
        'min_iou_vs_prev':  float(np.min(ious)),
        'mean_pos_frac':    float(np.mean(pos_fracs)),
        'mean_grow_added_px': float(np.mean(grow_added)),
    }
    return new_pseudo, diag

In [ ]:
def converged(diag, prev_pos_frac):
    # Stop only when self-consistency is high AND positive fraction is sane.
    iou_ok = (1.0 - diag['mean_iou_vs_prev']) < iter_cfg.stop_iou_eps
    pf = diag['mean_pos_frac']
    pf_change_ok = abs(pf - prev_pos_frac) < iter_cfg.stop_pos_frac_eps
    pf_bounded = (iter_cfg.pos_frac_min <= pf <= iter_cfg.pos_frac_max)
    return iou_ok and pf_change_ok and pf_bounded

## Outer loop

In [ ]:
csv_fields = [
    'iter_idx', 'epoch', 'phase',
    'train_loss', 'train_dice_loss', 'train_bce_loss',
    'val_loss', 'val_dice',
    'lr_encoder', 'lr_decoder',
    'epoch_time_s', 'train_time_s', 'val_time_s',
    'grad_norm_mean', 'grad_norm_max',
]
csv_logger = CSVMetricLogger(save_dir / 'epoch_metrics.csv', csv_fields)

iter_csv = CSVMetricLogger(
    save_dir / 'iter_metrics.csv',
    [
        'iter_idx', 'best_val_dice', 'mean_iou_vs_prev', 'min_iou_vs_prev',
        'mean_pos_frac', 'mean_grow_added_px', 'wall_time_s', 'stopped',
    ],
)

In [ ]:
if base_cfg.dry_run:
    logger.info('dry_run=True -- skipping iterative loop')
else:
    prev_pos_frac = float(np.mean([compute_positive_fraction(m) for m in current_pseudo.values()]))
    logger.info(f'starting iterative loop: max_iters={iter_cfg.max_iters}, '
                f'epochs_per_iter={seg_cfg.epochs}, prev_pos_frac={prev_pos_frac:.5f}')

    for it in range(1, iter_cfg.max_iters + 1):
        t_iter = time.time()
        logger.info(f'===== iteration {it}/{iter_cfg.max_iters} =====')

        if iter_cfg.reinit_model_per_iter:
            logger.info('reinit_model_per_iter=True -> rebuilding SwinUNETR')
            model = build_model()
        freeze_enc = iter_cfg.freeze_encoder_iter1_only and it == 1

        train_loader, val_loader = build_loaders(current_pseudo)

        best_val_dice = train_iteration(
            model, train_loader, val_loader,
            iter_idx=it, freeze_encoder=freeze_enc, csv_logger=csv_logger,
        )
        logger.info(f'iter {it} train done. best val dice (vs pseudo) = {best_val_dice:.4f}')

        # Save per-iteration checkpoint
        iter_ckpt = save_dir / f'iter{it:02d}_model.pt'
        torch.save({
            'iter_idx': it,
            'model_state_dict': model.state_dict(),
            'channel_mean': ch_mean.tolist(),
            'channel_std':  ch_std.tolist(),
            'best_val_dice_vs_pseudo': best_val_dice,
        }, iter_ckpt)

        # ---- refinement ----
        prob_maps = predict_per_patch_probs(model)
        new_pseudo, diag = refine_pseudo_labels(current_pseudo, prob_maps)
        diag['iter_idx'] = it
        diag['best_val_dice'] = best_val_dice
        diag['wall_time_s'] = time.time() - t_iter
        diag['stopped'] = False
        logger.info(
            f'iter {it} refinement: mean_iou_vs_prev={diag["mean_iou_vs_prev"]:.4f} '
            f'min_iou={diag["min_iou_vs_prev"]:.4f} '
            f'mean_pos_frac={diag["mean_pos_frac"]:.5f} '
            f'mean_grow_added_px={diag["mean_grow_added_px"]:.2f}'
        )

        stop = converged(diag, prev_pos_frac)
        diag['stopped'] = stop
        iter_csv.log(diag)

        # Persist pseudo-label tensor (stacked over the canonical record order)
        fnames_order = [r['filename'] for r in patch_ds.records]
        stack = np.stack([new_pseudo[f] for f in fnames_order], axis=0).astype(np.uint8)
        np.savez_compressed(
            save_dir / f'iter{it:02d}_pseudo.npz',
            filenames=np.array(fnames_order),
            masks=stack,
        )

        # Commit
        current_pseudo = new_pseudo
        prev_pos_frac = diag['mean_pos_frac']

        if stop:
            logger.info(f'converged at iter {it}; breaking')
            break
    else:
        logger.info(f'max_iters={iter_cfg.max_iters} reached without convergence')

## Iteration diagnostics

In [ ]:
import pandas as pd
ep_df  = pd.read_csv(save_dir / 'epoch_metrics.csv')
it_df  = pd.read_csv(save_dir / 'iter_metrics.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(it_df['iter_idx'], it_df['best_val_dice'], marker='o')
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('best val Dice (vs pseudo)')
axes[0].set_title('Val Dice per iteration')
axes[0].grid(alpha=0.3)

axes[1].plot(it_df['iter_idx'], it_df['mean_iou_vs_prev'], marker='o', color='tab:orange')
axes[1].axhline(1 - iter_cfg.stop_iou_eps, ls='--', color='gray',
                label=f'stop @ {1 - iter_cfg.stop_iou_eps:.3f}')
axes[1].set_xlabel('iteration'); axes[1].set_ylabel('mean IoU(prev, new)')
axes[1].set_title('Pseudo-label self-consistency')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(it_df['iter_idx'], it_df['mean_pos_frac'], marker='o', color='tab:green')
axes[2].set_xlabel('iteration'); axes[2].set_ylabel('mean positive-pixel fraction')
axes[2].set_title('Positive-fraction trajectory')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(save_dir / 'iter_metrics.png', dpi=120)
plt.show()

## Final full-image inference

In [ ]:
available = list_image_indices(
    data_cfg.data_root,
    exclude_patterns=data_cfg.exclude_patterns,
)
demo_idx = available[0]
full_image, _records = reassemble_image(
    data_cfg.data_root, demo_idx,
    exclude_patterns=data_cfg.exclude_patterns,
)
logger.info(f'demo image {demo_idx}: shape={full_image.shape}')

prob_map = sliding_window_predict(
    model, full_image,
    patch_size=model_cfg.img_size,
    ch_mean=ch_mean_t, ch_std=ch_std_t,
    device=device, overlap=0.5, batch_size=16,
)

plot_full_image_result(
    full_image, prob_map, pseudolabel=None,
    channel=0, threshold=0.5,
    title=f'Image {demo_idx}: final iterative model',
    save_to=save_dir / 'final_full_image.png',
)
plt.show()

## Save final model

In [ ]:
final_path = save_dir / seg_cfg.model_save_name
torch.save({
    'model_state_dict': model.state_dict(),
    'channel_mean': ch_mean.tolist(),
    'channel_std':  ch_std.tolist(),
    'iters_completed': int(it_df['iter_idx'].max()),
    'init_source': base_cfg.init_source,
    'method_name': base_cfg.method_name,
    'model_cfg': {
        'in_channels': model_cfg.in_channels,
        'img_size': model_cfg.img_size,
        'feature_size': model_cfg.feature_size,
        'depths': list(model_cfg.depths),
        'num_heads': list(model_cfg.num_heads),
    },
}, final_path)
logger.info(f'final model saved to {final_path}')

In [ ]:
logger.info('Done.')